In [ ]:
import boto3
import pandas as pd

s3 = boto3.client('s3')
bucket = 'cust-seg-ygp'
key = 'cleaned_online_retail_II.csv'

obj = s3.get_object(Bucket=bucket, Key=key)
df = pd.read_csv(obj['Body'])

df.head()

In [ ]:
print(df['InvoiceDate'].dtype)

object


In [ ]:
print(df['InvoiceDate'].head())

0    2009-12-01 07:45:00
1    2009-12-01 07:45:00
2    2009-12-01 07:45:00
3    2009-12-01 07:45:00
4    2009-12-01 07:45:00
Name: InvoiceDate, dtype: object


In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [ ]:
print(df['InvoiceDate'].head())

0   2009-12-01 07:45:00
1   2009-12-01 07:45:00
2   2009-12-01 07:45:00
3   2009-12-01 07:45:00
4   2009-12-01 07:45:00
Name: InvoiceDate, dtype: datetime64[ns]


In [ ]:
df['Customer ID'].isna().sum()

0

In [ ]:
# 'TotalPrice' to get monetary per line
df['TotalPrice'] = df['Price'] * df['Quantity']

In [ ]:
print(df['TotalPrice'].head(10))

0     83.4
1     81.0
2     81.0
3    100.8
4     30.0
5     39.6
6     30.0
7     59.5
8     30.6
9     45.0
Name: TotalPrice, dtype: float64


In [ ]:
# create a snapshot for recency
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
print(snapshot_date)

2011-12-10 12:50:00


In [ ]:
df['RecencyTemp'] = snapshot_date - df['InvoiceDate']

In [ ]:
df.head()

In [ ]:
# groupby
rfm = df.groupby('Customer ID').agg(
    Recency=('RecencyTemp', 'min'),
    Frequency=('Invoice', 'nunique'),
    Monetary=('TotalPrice', 'sum')
).reset_index()

In [ ]:
rfm.head()

In [ ]:
# convert days
rfm['Recency'] = rfm['Recency'].dt.days

In [ ]:
rfm.head()

In [ ]:
# less than 0 monetary
(rfm['Monetary'] < 0).sum()

0

In [ ]:
# actual columns less than 0 monetary
rfm[rfm['Monetary'] < 0]

In [ ]:
# now drop from *df*, not rfm
df = df.drop(columns='RecencyTemp')

In [ ]:
df.head()

In [ ]:
rfm.describe()

In [ ]:
import boto3
import io

# file name you want to save as
rfm_file_name = 'rfm_table.csv'

# convert RFM dataframe to CSV in memory
csv_buffer = io.StringIO()
rfm.to_csv(csv_buffer, index=False)

# upload to S3
s3 = boto3.client('s3')
bucket = 'cust-seg-ygp'   # your bucket name

s3.put_object(
    Bucket=bucket,
    Key=rfm_file_name,
    Body=csv_buffer.getvalue()
)

print(f"Saved RFM file to s3://{bucket}/{rfm_file_name}")

Saved RFM file to s3://cust-seg-ygp/rfm_table.csv
